In [ ]:
import sys
import importlib

!{sys.executable} -m pip install --no-cache-dir ultralytics matplotlib tqdm -q

!{sys.executable} -m pip uninstall -y opencv-python opencv-contrib-python opencv-python-headless opencv-contrib-python-headless 2>/dev/null
!{sys.executable} -m pip install --no-cache-dir opencv-python-headless -q

for mod in [k for k in sys.modules if k.startswith("cv2")]:
    del sys.modules[mod]
importlib.invalidate_caches()

import cv2
print(f"Using cv2 {cv2.__version__} from {cv2.__file__}")

In [ ]:
INPUT_VIDEO = "video.mp4"
OUTPUT_VIDEO = "tracked.mp4"
FRAMES_DIR = "frames"
CLEAN_OLD_ARTIFACTS = True
DETECT_THRESHOLD = 0.10
DISPLAY_THRESHOLD = 0.30
MAX_FRAMES = None
IMGSZ = 1920
ROTATE_DEGREES = -0.55
KEEP_COURT_SIDE = "near"
NET_FOOT_Y = 547
NET_FOOT_Y_RATIO = 0.5065
DRAW_SIDE_FILTER_LINE = True

In [ ]:
import torch

assert torch.cuda.is_available(), "CUDA not available — check your GPU instance"

device = torch.device("cuda")

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu_name}")
print(f"VRAM: {vram_gb:.1f} GB")
print(f"CUDA: {torch.version.cuda}")
print(f"PyTorch: {torch.__version__}")

In [ ]:
import gc
import os
import shutil

if CLEAN_OLD_ARTIFACTS:
    stale_names = [
        "frames",
        "tracking_results",
        "tracking_data",
        "output_json",
        "model",
    ]
    for name in stale_names:
        if name in globals():
            del globals()[name]

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

    stale_files = [
        OUTPUT_VIDEO,
        OUTPUT_VIDEO.replace(".mp4", "_tracks.json"),
    ]
    for path in stale_files:
        if os.path.exists(path):
            os.remove(path)
            print(f"Removed stale file: {path}")

    if os.path.isdir(FRAMES_DIR):
        shutil.rmtree(FRAMES_DIR)
        print(f"Removed stale folder: {FRAMES_DIR}/")

    print("Fresh-run cleanup complete.")
else:
    print("Fresh-run cleanup skipped (CLEAN_OLD_ARTIFACTS=False).")

In [ ]:
import cv2
import numpy as np
from PIL import Image
from tqdm import tqdm

cap = cv2.VideoCapture(INPUT_VIDEO)
assert cap.isOpened(), f"Cannot open: {INPUT_VIDEO}"

fps = cap.get(cv2.CAP_PROP_FPS)
orig_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
orig_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
if MAX_FRAMES:
    total_frames = min(total_frames, MAX_FRAMES)


PIL_BICUBIC = Image.Resampling.BICUBIC if hasattr(Image, "Resampling") else Image.BICUBIC


def build_rotation_params(width, height, angle_deg):
    pil_angle = -angle_deg
    mask = Image.new("RGBA", (width, height), (255, 255, 255, 255))
    rotated_mask = mask.rotate(pil_angle, resample=PIL_BICUBIC, expand=True, fillcolor=(0, 0, 0, 0))
    bound_w, bound_h = rotated_mask.size
    alpha = np.array(rotated_mask.getchannel("A"))
    valid = (alpha > 0).astype(np.uint8)
    integral = cv2.integral(valid)
    target_aspect = width / height

    def region_all_valid(x1, y1, crop_w, crop_h):
        x2 = x1 + crop_w
        y2 = y1 + crop_h
        total = integral[y2, x2] - integral[y1, x2] - integral[y2, x1] + integral[y1, x1]
        return total == crop_w * crop_h

    lo, hi = 1, bound_h
    best_w, best_h = width, height
    while lo <= hi:
        crop_h = (lo + hi) // 2
        crop_w = int(round(crop_h * target_aspect))
        if crop_w > bound_w:
            hi = crop_h - 1
            continue
        x1 = (bound_w - crop_w) // 2
        y1 = (bound_h - crop_h) // 2
        if region_all_valid(x1, y1, crop_w, crop_h):
            best_w, best_h = crop_w, crop_h
            lo = crop_h + 1
        else:
            hi = crop_h - 1

    return {
        "bound_w": bound_w,
        "bound_h": bound_h,
        "crop_x1": (bound_w - best_w) // 2,
        "crop_y1": (bound_h - best_h) // 2,
        "crop_w": best_w,
        "crop_h": best_h,
    }


def preprocess_frame(frame_bgr, params, output_size, angle_deg):
    if abs(float(angle_deg)) < 1e-6:
        return frame_bgr

    pil_angle = -float(angle_deg)
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    pil_image = Image.fromarray(frame_rgb)
    rotated = pil_image.rotate(pil_angle, resample=PIL_BICUBIC, expand=True, fillcolor=(0, 0, 0))
    x1 = params["crop_x1"]
    y1 = params["crop_y1"]
    crop_w = params["crop_w"]
    crop_h = params["crop_h"]
    cropped = rotated.crop((x1, y1, x1 + crop_w, y1 + crop_h))
    if cropped.size != output_size:
        cropped = cropped.resize(output_size, resample=PIL_BICUBIC)
    return cv2.cvtColor(np.array(cropped), cv2.COLOR_RGB2BGR)


_active_rotation = float(ROTATE_DEGREES)
rotation_params = build_rotation_params(orig_w, orig_h, _active_rotation)
FILTER_NET_FOOT_Y = NET_FOOT_Y
if FILTER_NET_FOOT_Y is None:
    FILTER_NET_FOOT_Y = int(round(orig_h * NET_FOOT_Y_RATIO))
FILTER_NET_FOOT_Y = max(0, min(orig_h - 1, int(FILTER_NET_FOOT_Y)))

print(f"Preprocessing config snapshot: ROTATE_DEGREES={_active_rotation}, rotation_params={rotation_params}")

frames = []
_rotation_applied_count = 0
for _ in tqdm(range(total_frames), desc="Loading frames"):
    ret, frame = cap.read()
    if not ret:
        break
    original_hash = frame.sum()
    frame = preprocess_frame(frame, rotation_params, (orig_w, orig_h), _active_rotation)
    if frame.sum() != original_hash:
        _rotation_applied_count += 1
    frames.append(frame)

cap.release()
print(f"Loaded {len(frames)} frames — {orig_w}x{orig_h} @ {fps:.1f} fps")
print(f"Rotation actually changed {_rotation_applied_count}/{len(frames)} frames (should be ALL if angle != 0)")
print(
    f"Rotation: {_active_rotation:.2f} deg | HTML-matched crop {rotation_params['crop_w']}x{rotation_params['crop_h']} -> resized back to {orig_w}x{orig_h}"
)

assert KEEP_COURT_SIDE in {"near", "far", "both"}, "KEEP_COURT_SIDE must be 'near', 'far', or 'both'"
print(
    f"Court-side filter: keeping {KEEP_COURT_SIDE} side | processed net-foot split y = {FILTER_NET_FOOT_Y}"
)


def keep_detection_for_side(box_xyxy, keep_side=KEEP_COURT_SIDE):
    if keep_side == "both":
        return True

    x1, y1, x2, y2 = [int(v) for v in box_xyxy]
    foot_y = y2

    if keep_side == "near":
        return foot_y >= FILTER_NET_FOOT_Y
    if keep_side == "far":
        return foot_y < FILTER_NET_FOOT_Y

    raise ValueError(f"Unknown keep_side: {keep_side}")


def filter_track_outputs(boxes_xyxy, track_ids, scores, keep_side=KEEP_COURT_SIDE):
    if boxes_xyxy is None or track_ids is None:
        return None, None, None

    keep_idx = [
        idx for idx in range(len(track_ids))
        if keep_detection_for_side(boxes_xyxy[idx].int().tolist(), keep_side)
    ]

    if not keep_idx:
        return None, None, None

    keep_idx = torch.tensor(keep_idx, dtype=torch.long)
    filtered_boxes = boxes_xyxy[keep_idx]
    filtered_ids = track_ids[keep_idx]
    filtered_scores = scores[keep_idx] if scores is not None else None
    return filtered_boxes, filtered_ids, filtered_scores

In [ ]:
import matplotlib.pyplot as plt

_preview_frame = frames[0].copy()

if KEEP_COURT_SIDE != "both":
    cv2.line(
        _preview_frame,
        (0, FILTER_NET_FOOT_Y),
        (_preview_frame.shape[1] - 1, FILTER_NET_FOOT_Y),
        (0, 0, 255),
        2,
    )
    side_label = f"NET_FOOT_Y = {FILTER_NET_FOOT_Y}  |  Keeping {KEEP_COURT_SIDE} court"
    cv2.putText(
        _preview_frame,
        side_label,
        (20, max(35, FILTER_NET_FOOT_Y - 12)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.9,
        (0, 0, 255),
        2,
        cv2.LINE_AA,
    )

rot_label = f"ROTATE_DEGREES = {ROTATE_DEGREES}"
cv2.putText(
    _preview_frame,
    rot_label,
    (20, _preview_frame.shape[0] - 30),
    cv2.FONT_HERSHEY_SIMPLEX,
    0.9,
    (255, 255, 255),
    2,
    cv2.LINE_AA,
)

plt.figure(figsize=(14, 8))
plt.imshow(cv2.cvtColor(_preview_frame, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title("Pre-Tracking Preview — Verify rotation + court dividing line before running YOLO")
plt.tight_layout()
plt.show()

del _preview_frame

In [ ]:
TRACKER_CONFIG = "botsort_reid.yaml"

tracker_yaml = """\
tracker_type: botsort
track_high_thresh: 0.3
track_low_thresh: 0.1
new_track_thresh: 0.65
track_buffer: 90
match_thresh: 0.75
fuse_score: true
gmc_method: none
proximity_thresh: 0.65
appearance_thresh: 0.8
with_reid: true
model: yolo26x-cls.pt
"""

with open(TRACKER_CONFIG, "w") as f:
    f.write(tracker_yaml)

print(f"Wrote {TRACKER_CONFIG}")
print(tracker_yaml)

In [ ]:
from ultralytics import YOLO

print("Loading YOLO26x detection model...")
model = YOLO("yolo26x.pt")
model.to(device)

_ = model.track(
    frames[0],
    persist=True,
    tracker=TRACKER_CONFIG,
    classes=[0],
    imgsz=IMGSZ,
    conf=DETECT_THRESHOLD,
    verbose=False,
)
model.predictor.trackers[0].reset()

vram_used = torch.cuda.memory_allocated() / 1e9
print(f"Model loaded + warm-up done — VRAM: {vram_used:.1f} / {vram_gb:.1f} GB")

In [ ]:
import time

print(f"Tracking {len(frames)} frames (person only, imgsz={IMGSZ})...")

tracking_results = {}
t0 = time.time()

for frame_idx, frame in enumerate(tqdm(frames, desc="Tracking")):
    results = model.track(
        frame,
        persist=True,
        tracker=TRACKER_CONFIG,
        classes=[0],
        imgsz=IMGSZ,
        conf=DETECT_THRESHOLD,
        verbose=False,
    )
    r = results[0]

    boxes_xyxy = None
    track_ids = None
    scores = None

    if r.boxes is not None and r.boxes.id is not None:
        boxes_xyxy = r.boxes.xyxy.cpu()
        track_ids = r.boxes.id.int().cpu()
        scores = r.boxes.conf.cpu()
        boxes_xyxy, track_ids, scores = filter_track_outputs(
            boxes_xyxy,
            track_ids,
            scores,
            keep_side=KEEP_COURT_SIDE,
        )

    tracking_results[frame_idx] = {
        "boxes_xyxy": boxes_xyxy,
        "track_ids": track_ids,
        "scores": scores,
    }

    if frame_idx % 100 == 0:
        torch.cuda.empty_cache()

elapsed = time.time() - t0
tracking_fps = len(frames) / elapsed
print(f"\nDone — {len(tracking_results)} frames in {elapsed:.1f}s ({tracking_fps:.1f} fps)")

if 0 in tracking_results and tracking_results[0]["track_ids"] is not None:
    ids = tracking_results[0]["track_ids"].tolist()
    print(f"Frame 0: {len(ids)} people, IDs: {ids}")

torch.cuda.empty_cache()

In [ ]:
COLORS = [
    (230, 25, 75),   (60, 180, 75),   (255, 225, 25),  (0, 130, 200),
    (245, 130, 48),  (145, 30, 180),  (70, 240, 240),  (240, 50, 230),
    (210, 245, 60),  (250, 190, 212), (0, 128, 128),   (220, 190, 255),
    (170, 110, 40),  (255, 250, 200), (128, 0, 0),     (170, 255, 195),
    (128, 128, 0),   (255, 215, 180), (0, 0, 128),     (128, 128, 128),
]


def get_color(obj_id):
    return COLORS[obj_id % len(COLORS)]


def draw_annotations(frame_bgr, boxes_xyxy, track_ids, scores):
    overlay = frame_bgr.copy()
    if DRAW_SIDE_FILTER_LINE and KEEP_COURT_SIDE != "both":
        cv2.line(overlay, (0, FILTER_NET_FOOT_Y), (overlay.shape[1] - 1, FILTER_NET_FOOT_Y), (255, 255, 255), 2)
        side_label = f"Keeping {KEEP_COURT_SIDE} court"
        cv2.putText(overlay, side_label, (20, max(35, FILTER_NET_FOOT_Y - 12)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2, cv2.LINE_AA)

    if track_ids is None or len(track_ids) == 0:
        return overlay, 0

    count = 0
    for i, tid in enumerate(track_ids.tolist()):
        x1, y1, x2, y2 = boxes_xyxy[i].int().tolist()
        score = float(scores[i]) if scores is not None else None
        if score is not None and score < DISPLAY_THRESHOLD:
            continue

        count += 1
        color = get_color(tid)

        cv2.rectangle(overlay, (x1, y1), (x2, y2), color, 2)

        label = f"ID:{tid}"
        if score is not None:
            label += f" {score:.2f}"
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.7, 2)
        cv2.rectangle(overlay, (x1, y1 - th - 10), (x1 + tw + 4, y1), color, -1)
        cv2.putText(overlay, label, (x1 + 2, y1 - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

    count_label = f"People: {count}"
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale, thickness = 1.0, 2
    (cw, ch), baseline = cv2.getTextSize(count_label, font, font_scale, thickness)
    pad = 12
    cx1 = overlay.shape[1] - cw - pad * 2 - 10
    cy1 = 10
    cx2 = overlay.shape[1] - 10
    cy2 = cy1 + ch + pad * 2
    sub = overlay[cy1:cy2, cx1:cx2].copy()
    cv2.rectangle(overlay, (cx1, cy1), (cx2, cy2), (0, 0, 0), -1)
    overlay[cy1:cy2, cx1:cx2] = cv2.addWeighted(sub, 0.35, overlay[cy1:cy2, cx1:cx2], 0.65, 0)
    cv2.rectangle(overlay, (cx1, cy1), (cx2, cy2), (255, 255, 255), 2)
    cv2.putText(overlay, count_label, (cx1 + pad, cy2 - pad),
                font, font_scale, (255, 255, 255), thickness, cv2.LINE_AA)

    return overlay, count

In [ ]:
print(f"Rendering {OUTPUT_VIDEO} ({orig_w}x{orig_h} @ {fps:.1f} fps)...")

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, fps, (orig_w, orig_h))

for frame_idx in tqdm(range(len(frames)), desc="Rendering"):
    frame_bgr = frames[frame_idx]
    tr = tracking_results[frame_idx]

    annotated, _ = draw_annotations(
        frame_bgr,
        tr["boxes_xyxy"],
        tr["track_ids"],
        tr["scores"],
    )

    writer.write(annotated)

writer.release()
print(f"Saved: {OUTPUT_VIDEO}")

In [ ]:
import matplotlib.pyplot as plt

preview_idx = 0
tr = tracking_results[preview_idx]

annotated, count = draw_annotations(
    frames[preview_idx],
    tr["boxes_xyxy"],
    tr["track_ids"],
    tr["scores"],
)

plt.figure(figsize=(14, 8))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title(f"Frame {preview_idx} — {count} people detected")
plt.tight_layout()
plt.show()

In [ ]:
import json

tracking_data = []

for frame_idx in sorted(tracking_results.keys()):
    tr = tracking_results[frame_idx]
    frame_dets = []

    if tr["track_ids"] is not None:
        for i, tid in enumerate(tr["track_ids"].tolist()):
            score = float(tr["scores"][i]) if tr["scores"] is not None else None
            if score is not None and score < DISPLAY_THRESHOLD:
                continue
            x1, y1, x2, y2 = tr["boxes_xyxy"][i].int().tolist()
            frame_dets.append({
                "id": int(tid),
                "bbox_xyxy": [x1, y1, x2, y2],
                "score": round(score, 4) if score is not None else None,
            })

    tracking_data.append({"frame": frame_idx, "detections": frame_dets})

output_json = OUTPUT_VIDEO.replace(".mp4", "_tracks.json")
with open(output_json, "w") as f:
    json.dump(tracking_data, f, indent=2)

unique_ids = set(d["id"] for fr in tracking_data for d in fr["detections"])
print(f"Saved: {output_json}")
print(f"Unique IDs: {len(unique_ids)} — {sorted(unique_ids)}")
print(f"Bboxes are in processed {orig_w}x{orig_h} coordinate space")
print(
    f"Court-side filter used: {KEEP_COURT_SIDE} | rotation = {ROTATE_DEGREES:.2f} deg | processed net-foot split y = {FILTER_NET_FOOT_Y}"
)

In [ ]:
import os

os.makedirs(FRAMES_DIR, exist_ok=True)

for idx, frame_bgr in enumerate(tqdm(frames, desc="Saving frames")):
    cv2.imwrite(os.path.join(FRAMES_DIR, f"{idx:06d}.png"), frame_bgr)

print(f"Saved {len(frames)} frames to {FRAMES_DIR}/")
print(f"Frame filenames match 'frame' field in {output_json}")